### Request commands dataset creation
v1.0: supports all the commands except for CRANK_REQUEST which is not supported even in the dashboard

In [1]:
import random
import json
import re

dataset = {}

commands = {
    "START": ["start.txt"],
    "STOP": ["stop.txt"],
    "ENGINE_START": ["engine_start.txt"],
    "ENGINE_STOP": ["engine_stop.txt"],
    "STOP_IN_PITLANE_TOGGLE_ON": ["stop_in_pitlane_toggle_on.txt"],
    "STOP_IN_PITLANE_TOGGLE_OFF": ["stop_in_pitlane_toggle_off.txt"],
    "SPEED_BY_RC_TOGGLE_ON": ["speed_by_rc_toggle_on.txt"],
    "SPEED_BY_RC_TOGGLE_OFF": ["speed_by_rc_toggle_off.txt"],
    "FLAG_BY_RC_TOGGLE_ON": ["flag_by_rc_toggle_on.txt"],
    "FLAG_BY_RC_TOGGLE_OFF": ["flag_by_rc_toggle_off.txt"],
    "INTO_PITLANE_TOGGLE_ON": ["into_pitlane_toggle_on.txt"],
    "INTO_PITLANE_TOGGLE_OFF": ["into_pitlane_toggle_off.txt"],
    "JOYSTICK_TOGGLE_ON": ["joystick_toggle_on.txt"],
    "JOYSTICK_TOGGLE_OFF": ["joystick_toggle_off.txt"],
    "SPEED_REQUEST": ["speed_request_mps.txt", "speed_request_kmh.txt"],
    "GG_SCALE": ["gg_scale.txt"],
    "OTHER": ["other.txt"],
}

In [2]:
for command_class, files in commands.items():
    for file_name in files:
        with open("../../data/enhanced_commands_augmentation/" + file_name) as file:
            lines = file.readlines()

            if command_class not in dataset:
                dataset[command_class] = {}

            for line in lines:
                line = line.strip()
                value = info = y = None
                # skip empty lines and lines starting with '---'
                if not line or line.startswith('---'):
                    continue

                if command_class in ["SPEED_REQUEST", "GG_SCALE"]:
                    line = line.split(",")
                    value = line[1].strip() if len(line) > 1 else None
                    info = line[2].strip() if len(line) > 2 else None
                    line = line[0]
                    if value is None or info is None:
                        continue
                    
                # remove non-alphanumeric characters and convert to lowercase
                line = re.sub(r'[^a-zA-Z0-9]', ' ', line).rstrip().lower()

                y = command_class + ("\n" + value if value else "") + ("\n" + info if info else "")

                # add the command to the dataset
                if line not in dataset[command_class]:
                    dataset[command_class][line] = y


##### Split data into train and test subsets and save them to JSON files

In [3]:
train_set = {}
test_set = {}
for command_class, commands in dataset.items():
    all_commands = list(commands.items())
    random.shuffle(all_commands)

    split_index = int(0.8 * len(all_commands))
    train_commands = all_commands[:split_index]
    test_commands = all_commands[split_index:]

    train_set[command_class] = dict(train_commands)
    test_set[command_class] = dict(test_commands)

with open("../../data/request_commands_dataset_v1.0.json", 'w') as file:
    json.dump(dataset, file, indent=4)
with open("../../data/request_commands_train_dataset_v1.0.json", 'w') as file:
    json.dump(train_set, file, indent=4)
with open("../../data/request_commands_test_dataset_v1.0.json", 'w') as file:
    json.dump(test_set, file, indent=4)